### Inspect Dataset Structure
Load only column names to avoid memory crash (887K rows × 74 columns).

In [1]:
import pandas as pd

df_header = pd.read_csv("../data/loan.csv", nrows=0)

print("Total columns:", len(df_header.columns))
print("\nAll column names:")
for i, col in enumerate(df_header.columns):
    print(f"{i}: {col}")

Total columns: 74

All column names:
0: id
1: member_id
2: loan_amnt
3: funded_amnt
4: funded_amnt_inv
5: term
6: int_rate
7: installment
8: grade
9: sub_grade
10: emp_title
11: emp_length
12: home_ownership
13: annual_inc
14: verification_status
15: issue_d
16: loan_status
17: pymnt_plan
18: url
19: desc
20: purpose
21: title
22: zip_code
23: addr_state
24: dti
25: delinq_2yrs
26: earliest_cr_line
27: inq_last_6mths
28: mths_since_last_delinq
29: mths_since_last_record
30: open_acc
31: pub_rec
32: revol_bal
33: revol_util
34: total_acc
35: initial_list_status
36: out_prncp
37: out_prncp_inv
38: total_pymnt
39: total_pymnt_inv
40: total_rec_prncp
41: total_rec_int
42: total_rec_late_fee
43: recoveries
44: collection_recovery_fee
45: last_pymnt_d
46: last_pymnt_amnt
47: next_pymnt_d
48: last_credit_pull_d
49: collections_12_mths_ex_med
50: mths_since_last_major_derog
51: policy_code
52: application_type
53: annual_inc_joint
54: dti_joint
55: verification_status_joint
56: acc_now_delinq


### Select Relevant Features
Keep 18 columns that are available at loan application time.
Dropped post-loan columns (total_pymnt, recoveries, etc.) to avoid data leakage.

In [2]:
import pandas as pd

keep_cols = [
    "loan_amnt", "term", "int_rate", "installment",
    "grade", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "purpose", "dti",
    "delinq_2yrs", "inq_last_6mths", "open_acc",
    "revol_util", "total_acc", "pub_rec",
    "loan_status"  # target
]

df = pd.read_csv("../data/loan.csv", usecols=keep_cols, low_memory=False)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (887379, 18)

First 5 rows:


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_util,total_acc
0,5000.0,36 months,10.65,162.87,B,10+ years,RENT,24000.0,Verified,Fully Paid,credit_card,27.65,0.0,1.0,3.0,0.0,83.7,9.0
1,2500.0,60 months,15.27,59.83,C,< 1 year,RENT,30000.0,Source Verified,Charged Off,car,1.00,0.0,5.0,3.0,0.0,9.4,4.0
2,2400.0,36 months,15.96,84.33,C,10+ years,RENT,12252.0,Not Verified,Fully Paid,small_business,8.72,0.0,2.0,2.0,0.0,98.5,10.0
3,10000.0,36 months,13.49,339.31,C,10+ years,RENT,49200.0,Source Verified,Fully Paid,other,20.00,0.0,1.0,10.0,0.0,21.0,37.0
4,3000.0,60 months,12.69,67.79,B,1 year,RENT,80000.0,Source Verified,Current,other,17.94,0.0,0.0,15.0,0.0,53.9,38.0


### Clean Target Variable
- 0 = Fully Paid (safe)
- 1 = Charged Off, Default, Late 31-120 days (risky)
- Removed unclear statuses (Current, Issued, Grace Period)

In [4]:
# All unique values in loan_status
print("Unique loan_status values:")
print(df["loan_status"].value_counts())

Unique loan_status values:
loan_status
Current                                                601779
Fully Paid                                             207723
Charged Off                                             45248
Late (31-120 days)                                      11591
Issued                                                   8460
In Grace Period                                          6253
Late (16-30 days)                                        2357
Does not meet the credit policy. Status:Fully Paid       1988
Default                                                  1219
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64


### Convert Text Features to Numeric
- term: "36 months" → 36
- emp_length: "10+ years" → 10
- grade: A→1, B→2, ..., G→7

In [5]:
good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]

df = df[df["loan_status"].isin(good + bad)]
df["loan_status"] = df["loan_status"].apply(lambda x: 1 if x in good else 0)

print("Shape after cleaning:", df.shape)
print("\nTarget Distribution:")
print(df["loan_status"].value_counts())

Shape after cleaning: (268530, 19)

Target Distribution:
loan_status
1    209711
0     58819
Name: count, dtype: int64


In [6]:

df["term"] = df["term"].str.replace(" months", "").astype(int)

print(df["term"].unique())

[36 60]


In [7]:
def clean_emp_length(val):
    if pd.isna(val):
        return 0
    val = str(val)
    if "10+" in val:
        return 10
    elif "< 1" in val:
        return 0
    else:
        return int(val.split()[0])

df["emp_length"] = df["emp_length"].apply(clean_emp_length)

print(df["emp_length"].unique())

[10  0  3  9  4  5  1  6  2  7  8]


In [8]:
grade_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
df["grade"] = df["grade"].map(grade_map)

print(df["grade"].unique())

[2 3 1 5 6 4 7]


In [9]:

print("Home Ownership:")
print(df["home_ownership"].value_counts())
print("\nPurpose:")
print(df["purpose"].value_counts())

Home Ownership:
home_ownership
MORTGAGE    131617
RENT        113191
OWN          23494
OTHER          179
NONE            48
ANY              1
Name: count, dtype: int64

Purpose:
purpose
debt_consolidation    157672
credit_card            52745
home_improvement       15852
other                  15361
major_purchase          6609
small_business          5116
car                     3790
medical                 3052
moving                  2175
wedding                 2019
house                   1764
vacation                1674
educational              422
renewable_energy         279
Name: count, dtype: int64


### One-Hot Encode Categorical Features
Convert home_ownership, purpose, verification_status into binary (0/1) columns.
Dropped rare home_ownership categories (OTHER, NONE, ANY).

In [10]:

df = df[df["home_ownership"].isin(["MORTGAGE", "RENT", "OWN"])]


df = pd.get_dummies(df, columns=["home_ownership", "purpose"], drop_first=True)

print("Shape after encoding:", df.shape)
print("\nAll columns now:")
print(df.columns.tolist())

Shape after encoding: (268302, 32)

All columns now:
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_util', 'total_acc', 'verification_status_Source Verified', 'verification_status_Verified', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding']


In [ ]:
df = pd.get_dummies(df, columns=["verification_status"], drop_first=True)

print("Final shape:", df.shape)
print(df.columns.tolist())

Final shape: (268302, 32)
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_util', 'total_acc', 'verification_status_Source Verified', 'verification_status_Verified', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding']


In [12]:
print(df.columns.tolist())

['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_util', 'total_acc', 'verification_status_Source Verified', 'verification_status_Verified', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding']


### Handle Missing Values
Fill remaining nulls with column median (< 0.1% of data affected).

In [14]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
loan_amnt                                0
term                                     0
int_rate                                 0
installment                              0
grade                                    0
emp_length                               0
annual_inc                               0
loan_status                              0
dti                                      0
delinq_2yrs                             25
inq_last_6mths                          25
open_acc                                25
pub_rec                                 25
revol_util                             243
total_acc                               25
verification_status_Source Verified      0
verification_status_Verified             0
home_ownership_OWN                       0
home_ownership_RENT                      0
purpose_credit_card                      0
purpose_debt_consolidation               0
purpose_educational                      0
purpose_home_improvement   

In [ ]:

cols_with_missing = ["delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "total_acc", "revol_util"]

for col in cols_with_missing:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after fix:")
print(df.isnull().sum().sum())

Missing values after fix:
0
